# Spin-Magnetic Phonon Density of States + Dispersions

Compute the phonon **dispersion curves** and phonon **density of states (DOS)** of a single material using Quantum ESPRESSO on the Mat3ra platform.

The workflow runs, in sequence: spin-polarized SCF (`pw.x`) → DFPT on a q-grid (`ph.x`) → interatomic force constants (`q2r.x`) → phonon DOS on a q-grid and dispersion along a high-symmetry path (`matdyn.x`).

Single material, loaded by name from `../uploads` or from the account's materials on the platform; no reference materials. Phonon DFPT cost scales steeply with the number of atoms, so use a **small primitive cell**.

For antiferromagnetic or otherwise labeled structures, assign numeric labels in Materials Designer to the atoms that need a separate starting magnetization (e.g. `Fe1`, `Fe2`) and leave the other atoms unlabeled. See the [Magnetic Properties (QE) tutorial](https://docs.mat3ra.com/guide/tutorials/dft/electronic/spin-magnetic-qe/).

<h2 style="color:green">Usage</h2>

1. Set the material and parameters in cells 1.2 / 1.3 below. Use a relaxed material, or set `ADD_RELAXATION = True` to relax it in the magnetic state. For harder cases (e.g. Graphene / 2D) use the controls in 1.3 — denser k/q grids, higher cutoffs — to avoid imaginary modes.
2. Click "Run" > "Run All".
3. Inspect the phonon dispersion curves and DOS at the end. Acoustic branches should go to 0 at Γ, with no large imaginary (negative) modes.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load the material.
4. Configure the Phonon DOS + Dispersions workflow.
5. Configure compute.
6. Create the job.
7. Submit and monitor the job.
8. Retrieve and visualize the phonon dispersion and DOS.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "FeO"

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "phonon_dos_dispersion.json"
MAGNETIC_WORKFLOW_SEARCH_TERM = "band_structure_magn.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Phonon Density of States + Dispersions (spin magnetism)"

# Model parameters
MODEL_SUBTYPE = "gga"  # "gga" or "lda"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1
TIME_LIMIT = "04:00:00"

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific phonon and spin-magnetic parameters

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft, GBRV [1]), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"           # for gga: "pbe", "pbesol"; for lda: "pz"

# Relax the structure before phonons. Phonons require a force-minimized geometry —
# an unrelaxed structure produces imaginary (negative) modes. Relaxes the atomic
# positions in the magnetic state, in the SCF step.
ADD_RELAXATION = False

# Electronic k-grids; if None, the workflow default is used
SCF_KGRID = None         # e.g. [8, 8, 8]

# Phonon q-grid for the DFPT run (ph.x); if None, the workflow default is used
PHONON_QGRID = None      # e.g. [4, 4, 4]

# DFPT mixing for magnetic metals (ph.x alpha_mix(1)); if None, the ph.x default (0.7) is used
ALPHA_MIX = 0.3

# Phonon DOS interpolation grid (matdyn.x); if None, the workflow default is used
DOS_QGRID = None         # e.g. [12, 12, 12]

# Phonon dispersion path (matdyn.x); if None, the workflow default is used
QPATH = None             # [{"point": "Γ", "steps": 20}, {"point": "N", "steps": 20}, {"point": "M", "steps": 20}, {"point": "Γ", "steps": 20}]

# Collinear magnetization by atomic species (with labels), e.g. antiferromagnetic FeO
STARTING_MAGNETIZATION = {"Fe1": -1, "Fe2": 1}

# Patch &CONTROL and &ELECTRONS (recommended for magnetic insulators)
ADDITIONAL_PARAMETERS = {
    # "control": {
    #     "tstress": False,
    #     "tprnfor": False,
    # },
    # "electrons": {
    #     "mixing_mode": "plain",
    #     "mixing_beta": 0.2,
    #     "mixing_ndim": 10,
    #     "conv_thr": "1.0d-8",
    #     "diagonalization": "cg",
    #     "electron_maxstep": 200,
    #     "diago_david_ndim": 4,
    #     "diago_full_acc": True,
    #     "startingwfc": "atomic+random",
    # }
}

# Energy cutoffs (raise for harder cases — e.g. 60 / 480 for Graphene)
ECUTWFC = 40
ECUTRHO = 200

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file or platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

material = load_material(client, FOLDER, MATERIAL_NAME, ACCOUNT_ID)

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

magnetic_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(MAGNETIC_WORKFLOW_SEARCH_TERM)
magnetic_scf_unit = Workflow.create(magnetic_workflow_config).subworkflows[0].get_unit_by_name(name="pw_scf_magn")
scf_unit = workflow.subworkflows[0].get_unit_by_name(name="pw_scf")
scf_unit.input = magnetic_scf_unit.input
workflow.subworkflows[0].set_unit(scf_unit)

visualize_workflow(workflow)

### 4.3. Set model and its parameters (physics)

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode import ModelFactory

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

visualize_workflow(workflow)

### 4.4. Modify method (computational parameters)
Optional overrides — each applies only if its parameter is set above. Electronic k-grid + cutoffs on `pw.x`; phonon q-grid and mixing (`alpha_mix`) on `ph.x`; DOS interpolation grid and dispersion path on `matdyn.x`. Starting magnetization and additional parameters are applied to the SCF `pw.x`.

In [ ]:
from mat3ra.wode.context.providers import (
    PlanewaveCutoffsContextProvider, PointsGridDataProvider, PointsPathDataProvider,
)
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input
import re

phonon_subworkflow = workflow.subworkflows[0]

# SCF electronic k-grid (pw.x)
if SCF_KGRID is not None:
    unit = phonon_subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(PointsGridDataProvider(material=material, dimensions=SCF_KGRID, isEdited=True).get_context_item_data())
    phonon_subworkflow.set_unit(unit)

# Phonon q-grid for DFPT (ph.x)
if PHONON_QGRID is not None:
    unit = phonon_subworkflow.get_unit_by_name(name="ph_grid")
    unit.add_context(PointsGridDataProvider(material=material, name="qgrid", dimensions=PHONON_QGRID, isEdited=True).get_context_item_data())
    phonon_subworkflow.set_unit(unit)

# DFPT mixing (ph.x)
if ALPHA_MIX is not None:
    patch_workflow_qe_input(workflow, {"inputph": {"alpha_mix(1)": ALPHA_MIX}}, unit_names=["ph_grid"])

# Phonon DOS interpolation grid (matdyn.x)
if DOS_QGRID is not None:
    unit = phonon_subworkflow.get_unit_by_name(name="matdyn_grid")
    unit.add_context(PointsGridDataProvider(material=material, name="igrid", dimensions=DOS_QGRID, isEdited=True).get_context_item_data())
    phonon_subworkflow.set_unit(unit)

# Phonon dispersion path (matdyn.x)
if QPATH is not None:
    unit = phonon_subworkflow.get_unit_by_name(name="matdyn_path")
    unit.add_context(PointsPathDataProvider(path=QPATH, isEdited=True, name="ipath").get_context_item_data())
    phonon_subworkflow.set_unit(unit)

# Energy cutoffs (pw.x)
if ECUTWFC is not None:
    unit = phonon_subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data())
    phonon_subworkflow.set_unit(unit)

# Build species names from material labels (e.g. Fe1, Fe2, O)
basis = material.basis
labels_map = {item["id"]: str(item["value"]) for item in basis.labels.to_dict()} if basis.labels else {}
species_names = []
seen = set()
for element in basis.elements.to_dict():
    label = labels_map.get(element["id"], "")
    name = f"{element['value']}{label}" if label else element["value"]
    if name not in seen:
        seen.add(name)
        species_names.append(name)

JINJA_MAGN_BLOCK = re.compile(
    r'\{%-?\s*if collinearMagnetization\.isTotalMagnetization.*?\{%-?\s*endif\s*%\}',
    re.DOTALL
)
unit = phonon_subworkflow.get_unit_by_name(name="pw_scf")
unit.input[0].template.content = JINJA_MAGN_BLOCK.sub("", unit.input[0].template.content)
phonon_subworkflow.set_unit(unit)

system_patch = {}
for atomic_species, value in STARTING_MAGNETIZATION.items():
    index = species_names.index(atomic_species) + 1
    system_patch[f"starting_magnetization({index})"] = value

patch_workflow_qe_input(workflow, {"system": system_patch}, unit_names=["pw_scf"])
if ADD_RELAXATION:
    patch_workflow_qe_input(workflow, {"control": {"calculation": "relax"}}, unit_names=["pw_scf"])
if ADDITIONAL_PARAMETERS:
    patch_workflow_qe_input(workflow, ADDITIONAL_PARAMETERS, unit_names=["pw_scf"])

### 4.5. Preview final workflow

In [ ]:
visualize_workflow(workflow)

### 4.6. Save workflow to collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, time limit: {TIME_LIMIT}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

job_name = f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print(f"✅ Job created: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Phonon dispersion

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

phonon_dispersions_data = get_properties_for_job(
    client, job_id, property_name=PropertyName.non_scalar.phonon_dispersions.value
)
visualize_properties(phonon_dispersions_data, title="Phonon Dispersion", extra_config={"material": material.to_dict()})

### 8.2. Phonon density of states

In [ ]:
phonon_dos_data = get_properties_for_job(
    client, job_id, property_name=PropertyName.non_scalar.phonon_dos.value
)
visualize_properties(phonon_dos_data, title="Phonon DOS", extra_config={"material": material.to_dict()})

## References

[1] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053